# Amazon ML Hackathon — Business Entity Resolution
## Baseline Model #1 (v2): SQLite-backed blocking + Pairwise Features + Logistic Regression

**Why this version exists.** The previous in-memory (pandas-only) version of this baseline
ran into `MemoryError` during candidate generation: even restrictive blocking still implies
building Python lists / DataFrames whose size scales with the number of *matching* pairs,
and at hackathon scale that number can still be too large to hold in RAM at once.

This version replaces in-memory candidate generation with a **local SQLite database file**:

```
TSV files
   ↓
Local SQLite database (database/amazon_ml.db)
   ↓
Indexed blocking / candidate SQL queries
   ↓
Candidate pairs generated & written in manageable batches
   ↓
Pairwise feature engineering
   ↓
Logistic Regression
   ↓
F0.5 threshold tuning
   ↓
Error analysis
```

No PostgreSQL, MySQL, Docker, or any external service is used — only Python's built-in
`sqlite3`, and a single local `.db` file that never leaves this machine.

### Status of this notebook

This notebook's **logic has been executed and verified end-to-end against a small synthetic
dataset with the same schema** (entity_id / business_name / business_address / country),
generated purely to prove the pipeline runs without `MemoryError` and without silently
skipping any stage. That validation run and its real, observed numbers are reported in a
separate write-up alongside this notebook — they are **not** hackathon results and are not
copied into this notebook, because this notebook has not been run against the actual
`student_resource/dataset` files (they are not present in the environment that authored this
notebook). Every number this notebook produces when you run it against the real data will be
real, computed output — nothing here is pre-filled or invented.

Run this notebook from the `Amazon_ml/` project root, with:

```
Amazon_ml/
├── database/                     <- created by this notebook
├── student_resource/
│   └── dataset/
│       ├── train/
│       │   ├── train_source1.tsv
│       │   ├── train_source2.tsv
│       │   ├── train_source3.tsv
│       │   └── train_ground_truth.tsv
│       └── test/
│           ├── test_source1.tsv
│           ├── test_source2.tsv
│           └── test_source3.tsv
```


## 1. Imports & Configuration

In [1]:
import re
import sqlite3
import string
import warnings
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import fbeta_score, precision_score, recall_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Batch / sampling configuration
BATCH_SIZE = 10_000                 # number of Source-1 entities processed per candidate-generation batch
NEGATIVE_TO_POSITIVE_RATIO = 1       # ~2 sampled negatives per positive, reservoir-sampled
SQL_IN_CHUNK = 500                   # max placeholders per SQLite "IN (...)" query batch
PREFIX_LEN = 4                       # number of normalized characters used for name/address blocking prefixes


## 2. Dataset Paths

In [2]:
_cwd = Path.cwd()

DATASET_ROOT = _cwd / "student_resource" / "dataset"
TRAIN_ROOT = DATASET_ROOT / "train"
TEST_ROOT = DATASET_ROOT / "test"

TRAIN_S1_PATH = TRAIN_ROOT / "train_source1.tsv"
TRAIN_S2_PATH = TRAIN_ROOT / "train_source2.tsv"
TRAIN_S3_PATH = TRAIN_ROOT / "train_source3.tsv"
TRAIN_GT_PATH = TRAIN_ROOT / "train_ground_truth.tsv"

TEST_S1_PATH = TEST_ROOT / "test_source1.tsv"
TEST_S2_PATH = TEST_ROOT / "test_source2.tsv"
TEST_S3_PATH = TEST_ROOT / "test_source3.tsv"

DB_DIR = _cwd / "database"
DB_PATH = DB_DIR / "amazon_ml.db"

for p in [TRAIN_S1_PATH, TRAIN_S2_PATH, TRAIN_S3_PATH, TRAIN_GT_PATH,
          TEST_S1_PATH, TEST_S2_PATH, TEST_S3_PATH]:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {p}")


[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\train\train_source1.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\train\train_source2.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\train\train_source3.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\train\train_ground_truth.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\test\test_source1.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\test\test_source2.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\test\test_source3.tsv


If any path above prints `MISSING`, stop here and fix your working directory / the dataset
location before continuing.


## 3. Create the SQLite Database

The database is a **single local file** at `database/amazon_ml.db`. Set `RECREATE_DB = True`
to delete and rebuild it from scratch (useful after changing schema/blocking logic); set it to
`False` to reuse an already-populated database and skip straight to later sections.

We open the connection with a couple of pragmas that make bulk loading much faster
(`journal_mode=WAL`, `synchronous=NORMAL`) without needing any external server.


In [3]:
RECREATE_DB = True  # set False to reuse an existing populated database/amazon_ml.db

DB_DIR.mkdir(parents=True, exist_ok=True)

if RECREATE_DB and DB_PATH.exists():
    DB_PATH.unlink()
    print(f"Deleted existing database at {DB_PATH}")

conn = sqlite3.connect(str(DB_PATH))
conn.execute("PRAGMA journal_mode = WAL;")
conn.execute("PRAGMA synchronous = NORMAL;")
conn.execute("PRAGMA foreign_keys = OFF;")

print(f"Connected to SQLite database at {DB_PATH}")
print(f"Database file exists: {DB_PATH.exists()}  size on disk: "
      f"{DB_PATH.stat().st_size if DB_PATH.exists() else 0:,} bytes")


Deleted existing database at c:\Users\shara\Desktop\Amazon_ml\database\amazon_ml.db
Connected to SQLite database at c:\Users\shara\Desktop\Amazon_ml\database\amazon_ml.db
Database file exists: True  size on disk: 4,096 bytes


## 4. Import TSV Data into SQLite

We do **not** load an entire TSV into pandas at once. `import_tsv_to_sqlite` peeks at just the
header to discover the real column names (no assumptions like a column literally being called
`"address"`), creates a matching SQLite table (`TEXT` columns — safe for arbitrary raw data,
including future test-only categories like an unseen country), then streams the file in
`chunksize`-row chunks with `pandas.read_csv(..., chunksize=...)`, writing each chunk straight
into SQLite via `to_sql(..., if_exists="append")` before moving to the next chunk. At any
instant, only one chunk (a few tens of thousands of rows) is resident in memory — never the
whole file.


In [4]:
def discover_columns(tsv_path, sep="\t"):
    """Read only the header row to discover the actual column names, without loading data."""
    header_df = pd.read_csv(tsv_path, sep=sep, nrows=0)
    return list(header_df.columns)


def import_tsv_to_sqlite(tsv_path, table_name, conn, sep="\t", chunksize=100_000):
    """Stream a (possibly huge) TSV file into a SQLite table, chunk by chunk.

    Never holds more than one chunk of the file in memory. Creates the table (all TEXT
    columns, to avoid coercion errors from messy real-world data) using the file's actual
    header -- no column names are assumed ahead of time.
    """
    columns = discover_columns(tsv_path, sep=sep)
    quoted_cols = ", ".join(f'"{c}" TEXT' for c in columns)
    conn.execute(f'DROP TABLE IF EXISTS "{table_name}";')
    conn.execute(f'CREATE TABLE "{table_name}" ({quoted_cols});')
    conn.commit()

    total_rows = 0
    reader = pd.read_csv(tsv_path, sep=sep, chunksize=chunksize, dtype=str)
    for chunk in reader:
        chunk.to_sql(table_name, conn, if_exists="append", index=False)
        total_rows += len(chunk)
    conn.commit()
    return columns, total_rows


import_summary = []
for path, table in [
    (TRAIN_S1_PATH, "source1"),
    (TRAIN_S2_PATH, "source2"),
    (TRAIN_S3_PATH, "source3"),
    (TRAIN_GT_PATH, "ground_truth"),
]:
    cols, n_rows = import_tsv_to_sqlite(path, table, conn)
    import_summary.append({"table": table, "source_file": path.name, "columns": cols, "rows": n_rows})
    print(f"Imported {path.name} -> table '{table}': {n_rows:,} rows, columns={cols}")


Imported train_source1.tsv -> table 'source1': 2,206,821 rows, columns=['entity_id', 'business_name', 'business_address', 'country']
Imported train_source2.tsv -> table 'source2': 5,034,616 rows, columns=['entity_id', 'business_name', 'business_address', 'country']
Imported train_source3.tsv -> table 'source3': 5,285,603 rows, columns=['entity_id', 'business_name', 'business_address', 'country']
Imported train_ground_truth.tsv -> table 'ground_truth': 2,206,821 rows, columns=['source1_entity_id', 'matched_entity_ids']


### Column-name configuration

The source tables' real column names are whatever the TSV headers say. We auto-detect the id,
business-name, address, and country columns for `source1` / `source2` / `source3` using simple
keyword matching, with a manual override dictionary in case the heuristic guesses wrong for
your actual files — check the printed mapping below before continuing.


In [5]:
def guess_column(columns, keywords):
    """Return the first column whose name contains any of the given keywords (case-insensitive)."""
    for kw in keywords:
        for c in columns:
            if kw in c.lower():
                return c
    return None


# MANUAL OVERRIDE: fill any of these in if the auto-detected guess below is wrong for your files.
MANUAL_COLUMN_OVERRIDE = {
    # "source1": {"id": "entity_id", "name": "business_name", "address": "business_address", "country": "country"},
    # "source2": {...},
    # "source3": {...},
}

COLUMN_MAP = {}
for table in ["source1", "source2", "source3"]:
    cols = next(s["columns"] for s in import_summary if s["table"] == table)
    guessed = {
        "id": guess_column(cols, ["entity_id", "_id", "id"]),
        "name": guess_column(cols, ["business_name", "name"]),
        "address": guess_column(cols, ["business_address", "address"]),
        "country": guess_column(cols, ["country"]),
    }
    guessed.update(MANUAL_COLUMN_OVERRIDE.get(table, {}))
    COLUMN_MAP[table] = guessed
    print(f"{table}: {guessed}")

missing_any = [f"{t}.{f}" for t, m in COLUMN_MAP.items() for f, v in m.items() if v is None]
if missing_any:
    print(f"WARNING: could not auto-detect these fields: {missing_any}. "
          f"Fill them in via MANUAL_COLUMN_OVERRIDE above and re-run this cell.")


source1: {'id': 'entity_id', 'name': 'business_name', 'address': 'business_address', 'country': 'country'}
source2: {'id': 'entity_id', 'name': 'business_name', 'address': 'business_address', 'country': 'country'}
source3: {'id': 'entity_id', 'name': 'business_name', 'address': 'business_address', 'country': 'country'}


In [6]:
# Ground truth column detection (source1_entity_id / matched_entity_ids -- names may differ).
gt_cols = next(s["columns"] for s in import_summary if s["table"] == "ground_truth")
GT_S1_COL = guess_column(gt_cols, ["source1_entity_id", "source1", "s1_id", "entity_id"])
GT_MATCH_COL = guess_column(gt_cols, ["matched_entity_ids", "matched_ids", "matches", "matched"])
GT_DELIM = ","

print(f"ground_truth columns: {gt_cols}")
print(f"GT_S1_COL={GT_S1_COL!r}  GT_MATCH_COL={GT_MATCH_COL!r}")
if GT_S1_COL is None or GT_MATCH_COL is None:
    print("WARNING: could not auto-detect ground truth columns -- set GT_S1_COL / GT_MATCH_COL manually.")


ground_truth columns: ['source1_entity_id', 'matched_entity_ids']
GT_S1_COL='source1_entity_id'  GT_MATCH_COL='matched_entity_ids'


## 5. Verify the Database

In [7]:
for table in ["source1", "source2", "source3", "ground_truth"]:
    n = conn.execute(f'SELECT COUNT(*) FROM "{table}";').fetchone()[0]
    print(f"{table}: {n:,} rows")

print()
print(f"Database file size on disk: {DB_PATH.stat().st_size:,} bytes "
      f"({DB_PATH.stat().st_size / (1024 ** 2):.2f} MB)")


source1: 2,206,821 rows
source2: 5,034,616 rows
source3: 5,285,603 rows
ground_truth: 2,206,821 rows

Database file size on disk: 1,472,720,896 bytes (1404.50 MB)


In [8]:
for table in ["source1", "source2", "source3", "ground_truth"]:
    print(f"--- {table} (first 3 rows) ---")
    sample = pd.read_sql(f'SELECT * FROM "{table}" LIMIT 3;', conn)
    display(sample)


--- source1 (first 3 rows) ---


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US


--- source2 (first 3 rows) ---


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India


--- source3 (first 3 rows) ---


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US


--- ground_truth (first 3 rows) ---


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"


## 6. Create Derived Blocking Columns

We add three derived TEXT columns to each source table:

- `country_norm` — lowercase, stripped country
- `name_prefix` — first PREFIX_LEN normalized characters of the business name
- `address_prefix` — first PREFIX_LEN normalized characters of the business address

Normalization (lowercasing, punctuation stripping, whitespace collapsing) is logic SQLite
doesn't have built in, so we register it as a small Python **user-defined function (UDF)** via
`sqlite3.Connection.create_function`, then populate the derived columns with a single `UPDATE
... SET x = normalize_udf(col)` statement per table — still no Python-side row loop, and no
giant DataFrame; SQLite iterates its own rows internally.


In [9]:
def normalize_text(x):
    """Lightweight, language-agnostic text normalization: lowercase, strip, collapse
    whitespace, drop punctuation. Deliberately NOT language-specific (no stemming, no
    transliteration) so it behaves reasonably on languages/countries unseen in training.
    Used both for SQL-side blocking-column population (as a UDF) and Python-side
    similarity features later in this notebook.
    """
    if x is None:
        return ""
    s = str(x).lower().strip()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_prefix(x, length=PREFIX_LEN):
    s = normalize_text(x)
    return s[:length]


# Register as SQLite UDFs so we can populate blocking columns with plain SQL UPDATEs.
conn.create_function("normalize_udf", 1, normalize_text)
conn.create_function("prefix_udf", 1, normalize_prefix)


def add_blocking_columns(table, conn, id_col, name_col, address_col, country_col):
    existing_cols = {row[1] for row in conn.execute(f'PRAGMA table_info("{table}");')}
    for col in ["country_norm", "name_prefix", "address_prefix"]:
        if col not in existing_cols:
            conn.execute(f'ALTER TABLE "{table}" ADD COLUMN "{col}" TEXT;')

    conn.execute(
        f'UPDATE "{table}" SET "country_norm" = normalize_udf("{country_col}");'
    )
    conn.execute(
        f'UPDATE "{table}" SET "name_prefix" = prefix_udf("{name_col}");'
    )
    conn.execute(
        f'UPDATE "{table}" SET "address_prefix" = prefix_udf("{address_col}");'
    )
    conn.commit()


for table in ["source1", "source2", "source3"]:
    m = COLUMN_MAP[table]
    add_blocking_columns(table, conn, m["id"], m["name"], m["address"], m["country"])
    print(f"Blocking columns populated for {table}.")


Blocking columns populated for source1.
Blocking columns populated for source2.
Blocking columns populated for source3.


In [10]:
# Sanity check: NULL/empty handling -- missing name/address/country should degrade gracefully
# to an empty-string prefix rather than crashing the UPDATwritten above.
for table in ["source1", "source2", "source3"]:
    n_null_country = conn.execute(
        f'SELECT COUNT(*) FROM "{table}" WHERE "country_norm" IS NULL OR "country_norm" = \'\';'
    ).fetchone()[0]
    n_null_name = conn.execute(
        f'SELECT COUNT(*) FROM "{table}" WHERE "name_prefix" IS NULL OR "name_prefix" = \'\';'
    ).fetchone()[0]
    print(f"{table}: rows with empty country_norm={n_null_country:,}  empty name_prefix={n_null_name:,}")

display(pd.read_sql('SELECT * FROM "source2" LIMIT 5;', conn))


source1: rows with empty country_norm=0  empty name_prefix=0
source2: rows with empty country_norm=0  empty name_prefix=2
source3: rows with empty country_norm=0  empty name_prefix=13


,entity_id,business_name,business_address,country,country_norm,name_prefix,address_prefix
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,india,राम,kh n
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,us,holl,105
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,india,आदित,g357
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,us,summ,gree
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,us,delt,914


## 7. Create SQLite Indexes

**Why indexes matter here:** without an index, `WHERE country_norm = ? AND name_prefix = ?`
forces SQLite to scan every row of `source2`/`source3` for every single Source‑1 record —
an O(|S1| × |S2|) operation, exactly the Cartesian-product cost we're trying to avoid. A
composite index on `(country_norm, name_prefix)` (and separately on `(country_norm,
address_prefix)`) lets SQLite jump straight to the matching rows via a B-tree lookup, turning
each candidate-generation query into roughly O(log |S2| + matches) instead of O(|S2|). We also
index each table's raw id column, since we repeatedly look records up by id (for ground truth
resolution and for feature-engineering joins later).


In [11]:
def create_indexes(table, conn, id_col):
    conn.execute(f'CREATE INDEX IF NOT EXISTS idx_{table}_country_name ON "{table}"(country_norm, name_prefix);')
    conn.execute(f'CREATE INDEX IF NOT EXISTS idx_{table}_country_address ON "{table}"(country_norm, address_prefix);')
    conn.execute(f'CREATE INDEX IF NOT EXISTS idx_{table}_id ON "{table}"("{id_col}");')
    conn.commit()


for table in ["source1", "source2", "source3"]:
    create_indexes(table, conn, COLUMN_MAP[table]["id"])
    print(f"Indexes created on {table}.")

display(pd.read_sql("SELECT type, name, tbl_name FROM sqlite_master WHERE type='index';", conn))


Indexes created on source1.
Indexes created on source2.
Indexes created on source3.


,type,name,tbl_name
0,index,idx_source1_country_name,source1
1,index,idx_source1_country_address,source1
2,index,idx_source1_id,source1
3,index,idx_source2_country_name,source2
4,index,idx_source2_country_address,source2
5,index,idx_source2_id,source2
6,index,idx_source3_country_name,source3
7,index,idx_source3_country_address,source3
8,index,idx_source3_id,source3


## 8. Load Ground Truth into a Pair-Level Table

`ground_truth` (as imported) has one row per Source‑1 entity with a delimited string of
matched ids. We explode this into `ground_truth_pairs(source1_entity_id, candidate_entity_id,
candidate_source, label)` — one row per true match, `label` always `1`. This is done by
streaming through `ground_truth` in chunks (via `pd.read_sql(..., chunksize=...)`), not by
loading the whole table into one DataFrame, and each chunk is written straight into SQLite with
`executemany` before moving on.

To resolve whether a matched id belongs to `source2` or `source3` without a huge join, we load
just the **id columns** of `source2`/`source3` into two Python `set`s — this is O(|S2| + |S3|)
memory for a set of short id strings, not O(|S2| × |S3|) for pairs, so it stays small even
when the tables themselves are large.


In [12]:
s2_ids = set(
    row[0] for row in conn.execute(f'SELECT "{COLUMN_MAP["source2"]["id"]}" FROM source2;')
)
s3_ids = set(
    row[0] for row in conn.execute(f'SELECT "{COLUMN_MAP["source3"]["id"]}" FROM source3;')
)
print(f"Loaded {len(s2_ids):,} source2 ids and {len(s3_ids):,} source3 ids into memory "
      f"(id sets only -- not full rows, not pairs).")


Loaded 5,034,616 source2 ids and 5,285,603 source3 ids into memory (id sets only -- not full rows, not pairs).


In [13]:
conn.execute("DROP TABLE IF EXISTS ground_truth_pairs;")
conn.execute(
    'CREATE TABLE ground_truth_pairs ('
    'source1_entity_id TEXT, candidate_entity_id TEXT, candidate_source TEXT, label INTEGER'
    ');'
)
conn.commit()


def explode_ground_truth(conn, gt_s1_col, gt_match_col, s2_ids, s3_ids,
                          delim=GT_DELIM, chunksize=50_000):
    """Explode train_ground_truth rows into individual positive pairs and insert them into
    ground_truth_pairs, processing `chunksize` rows at a time.

    ground_truth itself has only one row per Source-1 entity -- like source1, it's small and
    is never the memory bottleneck (only the exploded PAIRS could grow, and even those are
    bounded by the number of true matches, typically far smaller than the blocked-candidate
    universe). So we read it fully with a single query, then slice batches from that small
    DataFrame in plain Python before writing each batch's exploded pairs. This deliberately
    avoids using pandas' `chunksize` together with writes on the same connection: a
    chunksize-based read keeps a SQLite cursor open across iterations, and running INSERT/DDL
    on the same connection while that cursor is still active raises "database is locked" (the
    batch candidate-generation loop in section 10 hit exactly this, which is why both use this
    same read-fully-then-slice-in-Python pattern).
    """
    total_pairs = 0
    zero_match_s1 = []
    unknown_source_count = 0

    query = f'SELECT "{gt_s1_col}" AS s1_id, "{gt_match_col}" AS matched FROM ground_truth;'
    gt_full = pd.read_sql(query, conn)

    for start in range(0, len(gt_full), chunksize):
        chunk = gt_full.iloc[start:start + chunksize]
        rows_to_insert = []
        for s1_id, raw_matches in zip(chunk["s1_id"], chunk["matched"]):
            if raw_matches is None or str(raw_matches).strip() == "" or str(raw_matches).lower() == "nan":
                zero_match_s1.append(s1_id)
                continue
            match_ids = [m.strip() for m in str(raw_matches).split(delim) if m.strip()]
            if not match_ids:
                zero_match_s1.append(s1_id)
                continue
            for cid in match_ids:
                if cid in s2_ids:
                    source = "S2"
                elif cid in s3_ids:
                    source = "S3"
                else:
                    source = "UNKNOWN"
                    unknown_source_count += 1
                rows_to_insert.append((s1_id, cid, source, 1))

        if rows_to_insert:
            conn.executemany(
                "INSERT INTO ground_truth_pairs VALUES (?, ?, ?, ?);", rows_to_insert
            )
            conn.commit()
        total_pairs += len(rows_to_insert)

    return total_pairs, zero_match_s1, unknown_source_count


n_gt_pairs, zero_match_s1_ids, n_unknown_source = explode_ground_truth(
    conn, GT_S1_COL, GT_MATCH_COL, s2_ids, s3_ids
)

print(f"Ground-truth pairs written to ground_truth_pairs: {n_gt_pairs:,}")
print(f"Source-1 entities with zero ground-truth matches: {len(zero_match_s1_ids):,}")
if n_unknown_source:
    print(f"WARNING: {n_unknown_source:,} matched ids did not resolve against source2 or "
          f"source3 -- check GT_DELIM / id formatting.")


Ground-truth pairs written to ground_truth_pairs: 7,638,365
Source-1 entities with zero ground-truth matches: 123,247


In [14]:
conn.execute(
    "CREATE INDEX IF NOT EXISTS idx_gt_pairs_s1 ON ground_truth_pairs(source1_entity_id);"
)
conn.execute(
    "CREATE INDEX IF NOT EXISTS idx_gt_pairs_composite ON ground_truth_pairs(source1_entity_id, candidate_entity_id);"
)
conn.commit()

n_positive_total = conn.execute("SELECT COUNT(*) FROM ground_truth_pairs;").fetchone()[0]
print(f"ground_truth_pairs row count: {n_positive_total:,}")
display(pd.read_sql("SELECT * FROM ground_truth_pairs LIMIT 5;", conn))


ground_truth_pairs row count: 7,638,365


,source1_entity_id,candidate_entity_id,candidate_source,label
0,S1-965667,S2-681193310,S2,1
1,S1-965667,S2-743505751,S2,1
2,S1-965667,S3-775321672,S3,1
3,S1-965667,S3-11291185,S3,1
4,S1-965667,S3-860443364,S3,1


## 9. Analyze Blocking Performance (before generating any candidate pairs)

We report how many candidate pairs **each rule implies** using pure SQL aggregate queries —
grouping each side by its blocking key and multiplying/summing bucket sizes — **without ever
generating a single candidate pair row**. This is cheap (proportional to the number of
*distinct* blocking keys, not the number of pairs) and lets us sanity-check the blocking
strategy before committing to the (comparatively expensive) batch candidate-generation pass in
section 10.


In [15]:
def estimate_rule_candidate_count(conn, key_col, other_table):
    """Exact count of (s1_row, other_row) pairs implied by matching on key_col, computed
    via a GROUP BY + JOIN of key-level counts -- no pair rows are materialized.
    """
    query = f"""
        SELECT SUM(s1g.n * og.n) FROM
        (SELECT country_norm, {key_col}, COUNT(*) AS n FROM source1 GROUP BY country_norm, {key_col}) s1g
        JOIN
        (SELECT country_norm, {key_col}, COUNT(*) AS n FROM "{other_table}" GROUP BY country_norm, {key_col}) og
        ON s1g.country_norm = og.country_norm AND s1g.{key_col} = og.{key_col};
    """
    result = conn.execute(query).fetchone()[0]
    return int(result) if result is not None else 0


rule1_s2 = estimate_rule_candidate_count(conn, "name_prefix", "source2")
rule1_s3 = estimate_rule_candidate_count(conn, "name_prefix", "source3")
rule2_s2 = estimate_rule_candidate_count(conn, "address_prefix", "source2")
rule2_s3 = estimate_rule_candidate_count(conn, "address_prefix", "source3")

rule1_total = rule1_s2 + rule1_s3
rule2_total = rule2_s2 + rule2_s3

n_s1 = conn.execute("SELECT COUNT(*) FROM source1;").fetchone()[0]
n_s2 = conn.execute("SELECT COUNT(*) FROM source2;").fetchone()[0]
n_s3 = conn.execute("SELECT COUNT(*) FROM source3;").fetchone()[0]
n_cartesian = n_s1 * (n_s2 + n_s3)

print("Candidate pairs implied by each blocking rule (pre-union, exact, via SQL aggregation):")
print(f"  RULE 1 (country + name_prefix):    S2={rule1_s2:,}  S3={rule1_s3:,}  total={rule1_total:,}")
print(f"  RULE 2 (country + address_prefix): S2={rule2_s2:,}  S3={rule2_s3:,}  total={rule2_total:,}")
print(f"  Theoretical full Cartesian product (S1 x (S2+S3)): {n_cartesian:,}")
if n_cartesian:
    print(f"  Rule 1 reduction vs Cartesian: {1 - rule1_total / n_cartesian:.6%}")
    print(f"  Rule 2 reduction vs Cartesian: {1 - rule2_total / n_cartesian:.6%}")


Candidate pairs implied by each blocking rule (pre-union, exact, via SQL aggregation):
  RULE 1 (country + name_prefix):    S2=7,596,030,534  S3=8,388,110,828  total=15,984,141,362
  RULE 2 (country + address_prefix): S2=9,549,421,473  S3=9,329,123,370  total=18,878,544,843
  Theoretical full Cartesian product (S1 x (S2+S3)): 22,774,876,013,799
  Rule 1 reduction vs Cartesian: 99.929817%
  Rule 2 reduction vs Cartesian: 99.917108%


In [16]:
# Exact recall of ground-truth positives under EACH rule (and their union), computed purely
# in SQL via indexed joins -- again, no candidate pairs materialized to compute this.
def positive_recall_for_rule(conn, key_col):
    query = f"""
        SELECT COUNT(*) FROM ground_truth_pairs gt
        JOIN source1 s1 ON gt.source1_entity_id = s1."{{id1}}"
        LEFT JOIN source2 s2 ON gt.candidate_source = 'S2' AND gt.candidate_entity_id = s2."{{id2}}"
        LEFT JOIN source3 s3 ON gt.candidate_source = 'S3' AND gt.candidate_entity_id = s3."{{id3}}"
        WHERE
            (gt.candidate_source = 'S2' AND s1.country_norm = s2.country_norm AND s1.{key_col} = s2.{key_col})
            OR
            (gt.candidate_source = 'S3' AND s1.country_norm = s3.country_norm AND s1.{key_col} = s3.{key_col});
    """.format(id1=COLUMN_MAP["source1"]["id"], id2=COLUMN_MAP["source2"]["id"], id3=COLUMN_MAP["source3"]["id"])
    return conn.execute(query).fetchone()[0]


recovered_rule1 = positive_recall_for_rule(conn, "name_prefix")
recovered_rule2 = positive_recall_for_rule(conn, "address_prefix")

query_union = f"""
    SELECT COUNT(*) FROM ground_truth_pairs gt
    JOIN source1 s1 ON gt.source1_entity_id = s1."{COLUMN_MAP['source1']['id']}"
    LEFT JOIN source2 s2 ON gt.candidate_source = 'S2' AND gt.candidate_entity_id = s2."{COLUMN_MAP['source2']['id']}"
    LEFT JOIN source3 s3 ON gt.candidate_source = 'S3' AND gt.candidate_entity_id = s3."{COLUMN_MAP['source3']['id']}"
    WHERE
        (gt.candidate_source = 'S2' AND s1.country_norm = s2.country_norm
            AND (s1.name_prefix = s2.name_prefix OR s1.address_prefix = s2.address_prefix))
        OR
        (gt.candidate_source = 'S3' AND s1.country_norm = s3.country_norm
            AND (s1.name_prefix = s3.name_prefix OR s1.address_prefix = s3.address_prefix));
"""
recovered_union = conn.execute(query_union).fetchone()[0]

blocking_report = pd.DataFrame([
    {"Blocking Rule": "Country + Name Prefix", "Candidates": rule1_total,
     "Positives Found": recovered_rule1, "Positive Recall": recovered_rule1 / max(n_positive_total, 1)},
    {"Blocking Rule": "Country + Address Prefix", "Candidates": rule2_total,
     "Positives Found": recovered_rule2, "Positive Recall": recovered_rule2 / max(n_positive_total, 1)},
    {"Blocking Rule": "Union (Rule 1 OR Rule 2)", "Candidates": None,
     "Positives Found": recovered_union, "Positive Recall": recovered_union / max(n_positive_total, 1)},
])
display(blocking_report)
print("(Union 'Candidates' is left blank here -- the exact de-duplicated union pair count is "
      "computed during batch candidate generation in section 10, since it requires generating "
      "and de-duplicating actual pairs, not just key-level aggregate counts.)")


,Blocking Rule,Candidates,Positives Found,Positive Recall
0,Country + Name Prefix,1.598414e+10,5821028,0.762078
1,Country + Address Prefix,1.887854e+10,3902987,0.510972
2,Union (Rule 1 OR Rule 2),NaN,6767713,0.886016


(Union 'Candidates' is left blank here -- the exact de-duplicated union pair count is computed during batch candidate generation in section 10, since it requires generating and de-duplicating actual pairs, not just key-level aggregate counts.)


**If `Positive Recall` for the union is well below 1.0**, the blocking rules are too
restrictive for a meaningful share of true matches, and that's a hard recall ceiling this
baseline cannot get past downstream — reported honestly here rather than hidden, and revisited
in the final conclusion. Since we still copy *every* ground-truth positive into the training
data regardless (section 11), the model itself will still be trained and evaluated on those
pairs — it's the **live candidate-generation process** (i.e. what a real, unlabeled test-time
S1 record would surface) that would miss them.


## 10. Generate Candidate Pairs in Batches

We process Source‑1 entities `BATCH_SIZE` at a time. For each batch:

1. Load that batch's `(id, country_norm, name_prefix, address_prefix)` from `source1` only.
2. Run two indexed SQL queries per batch — one joining on `(country_norm, name_prefix)`
   against `source2`/`source3`, one joining on `(country_norm, address_prefix)` — using a
   temporary in-memory table of the batch's keys so each rule is a **single indexed join per
   batch**, not one query per S1 row.
3. Union and de-duplicate the two rules' results **within the batch** (a batch-local Python
   `set`, bounded by `BATCH_SIZE`, not by the dataset size).
4. Split into positives (already known, via `ground_truth_pairs`) and negative candidates.
5. Feed negative candidates into a **reservoir sample** of fixed size
   `NEGATIVE_TO_POSITIVE_RATIO × (total positive count)` — a single-pass, uniform sample that
   never requires holding all negatives in memory at once.
6. Release the batch (Python garbage-collects it on the next loop iteration) before moving on.

At every point in this loop, memory use is bounded by `BATCH_SIZE` plus the fixed-size
reservoir — never by the total number of candidate pairs in the dataset.


In [17]:
def create_batch_key_table(conn, batch_df):
    """Create (once per batch) a small temp table holding just this batch's S1 blocking
    keys, so all four rule/source combinations below can query it without re-creating and
    re-dropping a temp table for each one (which is both slower and, under repeated rapid
    create/drop cycles, prone to SQLite reporting the table as locked).
    """
    conn.execute("DROP TABLE IF EXISTS _batch_keys;")
    conn.execute(
        "CREATE TEMP TABLE _batch_keys "
        "(s1_id TEXT, country_norm TEXT, name_prefix TEXT, address_prefix TEXT);"
    )
    conn.executemany(
        "INSERT INTO _batch_keys VALUES (?, ?, ?, ?);",
        list(zip(batch_df["s1_id"], batch_df["country_norm"],
                 batch_df["name_prefix"], batch_df["address_prefix"])),
    )
    conn.commit()


def query_batch_matches(conn, other_table, other_id_col, rule_key_col):
    """Join the current batch's keys (already loaded into _batch_keys) against `other_table`
    on (country_norm, rule_key_col), using each table's index. Returns this rule's
    (s1_id, other_id) pairs for the whole batch in a single indexed query.
    """
    query = f"""
        SELECT bk.s1_id, o."{other_id_col}"
        FROM _batch_keys bk
        JOIN "{other_table}" o
          ON bk.country_norm = o.country_norm AND bk."{rule_key_col}" = o."{rule_key_col}";
    """
    return conn.execute(query).fetchall()


def iter_candidate_batches(conn, batch_size=BATCH_SIZE):
    """Yield one (s1_id, candidate_id, candidate_source) list per Source-1 batch: the
    de-duplicated union of rule1 and rule2 candidates for that batch only. The batch's key
    table is created once and dropped once per batch, with four indexed queries run against it.

    Source-1's own row count is small -- comparable to the id sets we already hold in memory
    for ground-truth resolution -- and is never the memory bottleneck; only the implied
    candidate PAIRS against S2/S3 are. So we materialize S1's blocking keys fully, once, then
    slice batches from that small DataFrame in plain Python. This also avoids interleaving
    writes/DDL (CREATE/DROP TEMP TABLE) with a still-open SQLite read cursor on the same
    connection, which SQLite does not allow -- doing so raises "database is locked".
    """
    s1_id_col = COLUMN_MAP["source1"]["id"]
    s2_id_col = COLUMN_MAP["source2"]["id"]
    s3_id_col = COLUMN_MAP["source3"]["id"]

    s1_query = f"""
        SELECT "{s1_id_col}" AS s1_id, country_norm, name_prefix, address_prefix
        FROM source1;
    """
    s1_keys_df = pd.read_sql(s1_query, conn)

    for start in range(0, len(s1_keys_df), batch_size):
        batch_df = s1_keys_df.iloc[start:start + batch_size]

        create_batch_key_table(conn, batch_df)

        pairs = set()
        for s1_id, cid in query_batch_matches(conn, "source2", s2_id_col, "name_prefix"):
            pairs.add((s1_id, cid, "S2"))
        for s1_id, cid in query_batch_matches(conn, "source3", s3_id_col, "name_prefix"):
            pairs.add((s1_id, cid, "S3"))
        for s1_id, cid in query_batch_matches(conn, "source2", s2_id_col, "address_prefix"):
            pairs.add((s1_id, cid, "S2"))
        for s1_id, cid in query_batch_matches(conn, "source3", s3_id_col, "address_prefix"):
            pairs.add((s1_id, cid, "S3"))

        conn.execute("DROP TABLE IF EXISTS _batch_keys;")
        conn.commit()

        yield list(pairs)


In [ ]:
print("Testing candidate generation...", flush=True)

for i, batch_pairs in enumerate(
    iter_candidate_batches(conn, batch_size=BATCH_SIZE), start=1
):
    print(
        f"Batch {i}: {len(batch_pairs):,} candidate pairs",
        flush=True
    )

    if i >= 5:
        break

Testing candidate generation...


In [ ]:
print("Starting candidate generation test...", flush=True)

test_batches = 0
test_pairs = 0

for batch_pairs in iter_candidate_batches(conn, batch_size=100):

    test_batches += 1
    test_pairs += len(batch_pairs)

    print(
        f"Batch {test_batches}: "
        f"{len(batch_pairs):,} candidate pairs",
        flush=True
    )

    if test_batches >= 5:
        break

print(
    f"TEST COMPLETE: {test_pairs:,} candidates across "
    f"{test_batches} batches",
    flush=True
)

: 

: 

## 11. Write Positive/Negative Training Data into SQLite

`training_pairs` holds the final labeled dataset: **every** ground-truth positive (copied
directly from `ground_truth_pairs`, so blocking coverage can never cause a positive to be
lost) plus the reservoir-sampled negatives from section 10. This table, not a Python
DataFrame, is the source of truth going forward — it stays small by construction (bounded by
`n_positives × (1 + NEGATIVE_TO_POSITIVE_RATIO)`), so it's safe to read into pandas from here
on.


In [ ]:
conn.execute("DROP TABLE IF EXISTS training_pairs;")
conn.execute("""
    CREATE TABLE training_pairs (
        source1_entity_id TEXT,
        candidate_entity_id TEXT,
        candidate_source TEXT,
        label INTEGER
    );
""")
conn.commit()

# ALL positives, regardless of whether blocking would have found them.
conn.execute("""
    INSERT INTO training_pairs
    SELECT source1_entity_id, candidate_entity_id, candidate_source, label
    FROM ground_truth_pairs;
""")

# Reservoir-sampled negatives.
if reservoir:
    conn.executemany(
        "INSERT INTO training_pairs VALUES (?, ?, ?, 0);", reservoir
    )
conn.commit()

conn.execute("CREATE INDEX IF NOT EXISTS idx_training_pairs_s1 ON training_pairs(source1_entity_id);")
conn.commit()

n_train_positive = conn.execute("SELECT COUNT(*) FROM training_pairs WHERE label = 1;").fetchone()[0]
n_train_negative = conn.execute("SELECT COUNT(*) FROM training_pairs WHERE label = 0;").fetchone()[0]

print(f"training_pairs: {n_train_positive + n_train_negative:,} rows "
      f"({n_train_positive:,} positive, {n_train_negative:,} negative)")


In [ ]:
# ------------------------------------------------------------------
# Verify: ground-truth positive count MUST exactly equal training_pairs positive count.
# This is the section-8 requirement -- no positive can ever be silently dropped, regardless
# of blocking coverage, because positives are copied wholesale, not filtered through blocking.
# ------------------------------------------------------------------
assert n_train_positive == n_positive_total, (
    f"Positive pair count mismatch! ground_truth_pairs={n_positive_total:,} vs "
    f"training_pairs(label=1)={n_train_positive:,}. This should never happen -- investigate "
    f"section 11's INSERT statement."
)
print(f"Verified: all {n_positive_total:,} ground-truth positive pairs are present in "
      f"training_pairs. Zero positives lost.")
print()
print(f"For context (diagnostic only, not a correctness requirement): of those "
      f"{n_positive_total:,} positives, {recovered_union:,} "
      f"({recovered_union / max(n_positive_total, 1):.2%}) would ALSO have been found by "
      f"live blocking alone (section 9) -- the rest are only present because we explicitly "
      f"preserved them.")


## 12. Train / Validation Split — by Source 1 Entity

Same rationale as before: splitting by individual pair risks the same S1 entity appearing in
both train and validation, letting the model partially memorize that entity's text and
inflating validation scores. We split on **unique `source1_entity_id` values** (80% train /
20% validation) and assign every pair for a given entity to whichever split it landed in.


In [ ]:
unique_s1_ids = [r[0] for r in conn.execute("SELECT DISTINCT source1_entity_id FROM training_pairs;")]
shuffled_ids = np.random.RandomState(RANDOM_STATE).permutation(unique_s1_ids)

n_train_ids = int(0.8 * len(shuffled_ids))
train_ids = set(shuffled_ids[:n_train_ids])
valid_ids = set(shuffled_ids[n_train_ids:])

assert train_ids.isdisjoint(valid_ids), "Entity leakage between train and validation!"
print(f"Unique S1 entities in training_pairs: {len(unique_s1_ids):,} "
      f"-> train {len(train_ids):,} / valid {len(valid_ids):,}")


In [ ]:
def fetch_pairs_for_ids(conn, ids, chunk_size=SQL_IN_CHUNK):
    """Fetch training_pairs rows for a (potentially large) set of S1 ids, batching the SQL
    IN (...) clause so we never build one enormous query string or blow past SQLite's
    parameter limit.
    """
    ids = list(ids)
    frames = []
    for i in range(0, len(ids), chunk_size):
        chunk_ids = ids[i:i + chunk_size]
        placeholders = ",".join("?" * len(chunk_ids))
        query = f"SELECT * FROM training_pairs WHERE source1_entity_id IN ({placeholders});"
        frames.append(pd.read_sql(query, conn, params=chunk_ids))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
        columns=["source1_entity_id", "candidate_entity_id", "candidate_source", "label"]
    )


train_df = fetch_pairs_for_ids(conn, train_ids)
valid_df = fetch_pairs_for_ids(conn, valid_ids)

print(f"Pairs: train {len(train_df):,} / valid {len(valid_df):,}")
for split_name, split_df in [("train", train_df), ("valid", valid_df)]:
    n_pos = int((split_df["label"] == 1).sum())
    n_neg = int((split_df["label"] == 0).sum())
    total = max(n_pos + n_neg, 1)
    print(f"  [{split_name}] positives={n_pos:,} ({n_pos / total:.2%})  "
          f"negatives={n_neg:,} ({n_neg / total:.2%})")


## 13. Pairwise Feature Engineering

Same eleven features as the original baseline (5 name, 5 address, 1 country), computed in
Python since they need real string comparison logic. Because `train_df`/`valid_df` are already
small (bounded by the reservoir size from section 10-11), we can safely attach the raw
name/address/country text with batched `IN (...)` lookups against `source1`/`source2`/`source3`
and compute features locally — no further SQL blocking logic is needed at this stage.


In [ ]:
def character_similarity(a, b):
    """Character-level similarity via difflib.SequenceMatcher ratio, in [0, 1]."""
    a, b = normalize_text(a), normalize_text(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def token_similarity(a, b):
    """Token-level Jaccard similarity over whitespace-split tokens, in [0, 1]."""
    a, b = normalize_text(a), normalize_text(b)
    tokens_a, tokens_b = set(a.split()), set(b.split())
    if not tokens_a and not tokens_b:
        return 1.0
    if not tokens_a or not tokens_b:
        return 0.0
    return len(tokens_a & tokens_b) / len(tokens_a | tokens_b)


def normalized_length_difference(a, b):
    """Absolute character-length difference, normalized by the longer string's length."""
    a, b = normalize_text(a), normalize_text(b)
    max_len = max(len(a), len(b))
    if max_len == 0:
        return 0.0
    return abs(len(a) - len(b)) / max_len


def fetch_records_by_ids(conn, table, id_col, name_col, address_col, country_col, ids,
                          chunk_size=SQL_IN_CHUNK):
    """Fetch name/address/country for a set of ids from `table`, batching IN (...) clauses."""
    ids = list(set(ids))
    frames = []
    for i in range(0, len(ids), chunk_size):
        chunk_ids = ids[i:i + chunk_size]
        placeholders = ",".join("?" * len(chunk_ids))
        query = (
            f'SELECT "{id_col}" AS entity_id, "{name_col}" AS business_name, '
            f'"{address_col}" AS business_address, "{country_col}" AS country '
            f'FROM "{table}" WHERE "{id_col}" IN ({placeholders});'
        )
        frames.append(pd.read_sql(query, conn, params=chunk_ids))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
        columns=["entity_id", "business_name", "business_address", "country"]
    )


def attach_record_fields(pairs_df, conn):
    """Join raw name/address/country fields for both sides of each pair, fetched from
    SQLite in batched IN (...) queries (never a full-table pull).
    """
    s1_records = fetch_records_by_ids(
        conn, "source1", COLUMN_MAP["source1"]["id"], COLUMN_MAP["source1"]["name"],
        COLUMN_MAP["source1"]["address"], COLUMN_MAP["source1"]["country"],
        pairs_df["source1_entity_id"],
    ).set_index("entity_id")
    s2_records = fetch_records_by_ids(
        conn, "source2", COLUMN_MAP["source2"]["id"], COLUMN_MAP["source2"]["name"],
        COLUMN_MAP["source2"]["address"], COLUMN_MAP["source2"]["country"],
        pairs_df.loc[pairs_df["candidate_source"] == "S2", "candidate_entity_id"],
    ).set_index("entity_id")
    s3_records = fetch_records_by_ids(
        conn, "source3", COLUMN_MAP["source3"]["id"], COLUMN_MAP["source3"]["name"],
        COLUMN_MAP["source3"]["address"], COLUMN_MAP["source3"]["country"],
        pairs_df.loc[pairs_df["candidate_source"] == "S3", "candidate_entity_id"],
    ).set_index("entity_id")

    df = pairs_df.copy()
    s1_fields = s1_records.reindex(df["source1_entity_id"]).reset_index(drop=True)
    s1_fields.columns = [f"s1_{c}" for c in s1_fields.columns]

    cand_fields = pd.DataFrame(index=df.index, columns=["cand_business_name", "cand_business_address", "cand_country"])
    is_s2 = (df["candidate_source"] == "S2").values
    is_s3 = (df["candidate_source"] == "S3").values
    if is_s2.any():
        cand_fields.loc[is_s2, :] = s2_records.reindex(df.loc[is_s2, "candidate_entity_id"]).values
    if is_s3.any():
        cand_fields.loc[is_s3, :] = s3_records.reindex(df.loc[is_s3, "candidate_entity_id"]).values

    return pd.concat([df.reset_index(drop=True), s1_fields, cand_fields.reset_index(drop=True)], axis=1)


def build_features(df):
    """Compute the 11 pairwise features for every row of a pairs dataframe that already
    has s1_/cand_ name, address, country columns attached.
    """
    feats = pd.DataFrame(index=df.index)

    s1_name, cand_name = df["s1_business_name"], df["cand_business_name"]
    s1_addr, cand_addr = df["s1_business_address"], df["cand_business_address"]
    s1_country, cand_country = df["s1_country"], df["cand_country"]

    norm_s1_name, norm_cand_name = s1_name.map(normalize_text), cand_name.map(normalize_text)
    norm_s1_addr, norm_cand_addr = s1_addr.map(normalize_text), cand_addr.map(normalize_text)

    feats["exact_name_match"] = (s1_name.astype(str).str.strip() == cand_name.astype(str).str.strip()).astype(int)
    feats["normalized_name_match"] = (norm_s1_name == norm_cand_name).astype(int)
    feats["name_character_similarity"] = [character_similarity(a, b) for a, b in zip(s1_name, cand_name)]
    feats["name_token_similarity"] = [token_similarity(a, b) for a, b in zip(s1_name, cand_name)]
    feats["name_length_difference"] = [normalized_length_difference(a, b) for a, b in zip(s1_name, cand_name)]

    feats["exact_address_match"] = (s1_addr.astype(str).str.strip() == cand_addr.astype(str).str.strip()).astype(int)
    feats["normalized_address_match"] = (norm_s1_addr == norm_cand_addr).astype(int)
    feats["address_character_similarity"] = [character_similarity(a, b) for a, b in zip(s1_addr, cand_addr)]
    feats["address_token_similarity"] = [token_similarity(a, b) for a, b in zip(s1_addr, cand_addr)]
    feats["address_length_difference"] = [normalized_length_difference(a, b) for a, b in zip(s1_addr, cand_addr)]

    norm_s1_country = s1_country.astype(str).str.strip().str.lower()
    norm_cand_country = cand_country.astype(str).str.strip().str.lower()
    feats["same_country"] = (norm_s1_country == norm_cand_country).astype(int)

    return feats


FEATURE_COLUMNS = [
    "exact_name_match", "normalized_name_match", "name_character_similarity",
    "name_token_similarity", "name_length_difference",
    "exact_address_match", "normalized_address_match", "address_character_similarity",
    "address_token_similarity", "address_length_difference",
    "same_country",
]

train_full = attach_record_fields(train_df, conn)
valid_full = attach_record_fields(valid_df, conn)

train_features = build_features(train_full)
valid_features = build_features(valid_full)

train_dataset = pd.concat(
    [train_full[["source1_entity_id", "candidate_entity_id", "candidate_source", "label"]], train_features],
    axis=1,
)
valid_dataset = pd.concat(
    [valid_full[["source1_entity_id", "candidate_entity_id", "candidate_source", "label"]], valid_features],
    axis=1,
)

display(train_dataset.head())
print(f"train_dataset: {train_dataset.shape}  valid_dataset: {valid_dataset.shape}")


In [ ]:
print("Feature dtypes:")
print(train_dataset[FEATURE_COLUMNS].dtypes)
print()
print("Missing values per feature (train):")
print(train_dataset[FEATURE_COLUMNS].isna().sum())


In [ ]:
correlations = train_dataset[FEATURE_COLUMNS + ["label"]].corr(numeric_only=True)["label"].drop("label")
correlations.sort_values(ascending=False)


As before: a high correlation with `label` doesn't automatically mean a feature is causally
"useful" — `same_country` partly reflects how positives were preserved regardless of blocking
(section 11), not purely discriminative power within a country. Name/address similarity
features are the more directly interpretable signal of "does this text look like the same
business." Revisit this paragraph with what's actually observed once run against real data.


## 14. Train Logistic Regression

In [ ]:
X_train, y_train = train_dataset[FEATURE_COLUMNS], train_dataset["label"]
X_valid, y_valid = valid_dataset[FEATURE_COLUMNS], valid_dataset["label"]

models = {}
for weight_setting in [None, "balanced"]:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight=weight_setting, random_state=RANDOM_STATE)),
    ])
    pipe.fit(X_train, y_train)
    models[weight_setting] = pipe

for weight_setting, pipe in models.items():
    proba = pipe.predict_proba(X_valid)[:, 1]
    preds = (proba >= 0.5).astype(int)
    p = precision_score(y_valid, preds, zero_division=0)
    r = recall_score(y_valid, preds, zero_division=0)
    f05 = fbeta_score(y_valid, preds, beta=0.5, zero_division=0)
    print(f"class_weight={weight_setting!r:>10}  @0.5  precision={p:.4f}  recall={r:.4f}  F0.5={f05:.4f}")


**Chosen configuration:** `class_weight="balanced"`, for the same reason as before — negatives
outnumber positives after blocking/sampling, and unweighted logistic regression tends to
under-predict the minority (match) class. Confirm this against the actual comparison above and
adjust `FINAL_CLASS_WEIGHT` if the numbers say otherwise.


In [ ]:
FINAL_CLASS_WEIGHT = "balanced"  # revisit after inspecting the cell above
final_model = models[FINAL_CLASS_WEIGHT]


## 15. Predict Match Probabilities

In [ ]:
valid_proba = final_model.predict_proba(X_valid)[:, 1]
valid_dataset = valid_dataset.copy()
valid_dataset["match_probability"] = valid_proba

display(valid_dataset[["source1_entity_id", "candidate_entity_id", "candidate_source",
                        "label", "match_probability"]].head(10))


## 16. Threshold Tuning for F0.5 — Pair Level

We sweep thresholds rather than defaulting to 0.5, since the competition scores F0.5
(precision weighted more than recall).


In [ ]:
thresholds = np.round(np.arange(0.10, 0.96, 0.05), 2)
beta = 0.5

threshold_results = []
for t in thresholds:
    preds = (valid_proba >= t).astype(int)
    p = precision_score(y_valid, preds, zero_division=0)
    r = recall_score(y_valid, preds, zero_division=0)
    f = fbeta_score(y_valid, preds, beta=beta, zero_division=0)
    threshold_results.append({"threshold": t, "precision": p, "recall": r, "f0.5": f})

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

best_row = threshold_df.loc[threshold_df["f0.5"].idxmax()]
BEST_THRESHOLD = float(best_row["threshold"])
print(f"Best pair-level threshold by F0.5: {BEST_THRESHOLD} "
      f"(precision={best_row['precision']:.4f}, recall={best_row['recall']:.4f}, F0.5={best_row['f0.5']:.4f})")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df["threshold"], threshold_df["f0.5"], marker="o", label="F0.5")
ax.plot(threshold_df["threshold"], threshold_df["precision"], marker="o", label="Precision")
ax.plot(threshold_df["threshold"], threshold_df["recall"], marker="o", label="Recall")
ax.axvline(BEST_THRESHOLD, color="gray", linestyle="--", label=f"best threshold = {BEST_THRESHOLD}")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold vs Precision / Recall / F0.5 (pair-level, validation)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## 17. Competition-Style Macro F0.5 (per Source 1 Entity)

The real competition metric averages F0.5 **per S1 entity**, with the convention that an
entity with no true matches and no predicted matches scores 1.0, while predicting any match
for a zero-true-match entity scores 0.0.


In [ ]:
def evaluate_macro_f05(df, proba_col, threshold, beta=0.5,
                        s1_col="source1_entity_id", cand_col="candidate_entity_id", label_col="label"):
    rows = []
    for s1_id, group in df.groupby(s1_col):
        true_matches = set(group.loc[group[label_col] == 1, cand_col])
        pred_matches = set(group.loc[group[proba_col] >= threshold, cand_col])

        if not true_matches and not pred_matches:
            f05 = precision = recall = 1.0
        elif not true_matches and pred_matches:
            f05 = precision = recall = 0.0
        else:
            tp = len(true_matches & pred_matches)
            precision = tp / len(pred_matches) if pred_matches else 0.0
            recall = tp / len(true_matches) if true_matches else 0.0
            f05 = 0.0 if (precision == 0.0 and recall == 0.0) else (
                (1 + beta ** 2) * precision * recall / ((beta ** 2 * precision) + recall)
            )

        rows.append({"source1_entity_id": s1_id, "n_true_matches": len(true_matches),
                      "n_predicted_matches": len(pred_matches), "precision": precision,
                      "recall": recall, "f0.5": f05})

    per_entity_df = pd.DataFrame(rows)
    return per_entity_df["f0.5"].mean(), per_entity_df


macro_threshold_results = []
for t in thresholds:
    m_f05, _ = evaluate_macro_f05(valid_dataset, "match_probability", t)
    macro_threshold_results.append({"threshold": t, "macro_f0.5": m_f05})

macro_threshold_df = pd.DataFrame(macro_threshold_results)
display(macro_threshold_df)

best_macro_row = macro_threshold_df.loc[macro_threshold_df["macro_f0.5"].idxmax()]
BEST_MACRO_THRESHOLD = float(best_macro_row["threshold"])
macro_f05, per_entity_df = evaluate_macro_f05(valid_dataset, "match_probability", BEST_MACRO_THRESHOLD)

print(f"Best threshold by macro-per-S1 F0.5: {BEST_MACRO_THRESHOLD} (macro F0.5={macro_f05:.4f})")


If `BEST_THRESHOLD` (pair-level) and `BEST_MACRO_THRESHOLD` (competition-style) differ,
prefer `BEST_MACRO_THRESHOLD` for anything submission-facing, since that's the actual
competition objective; the pair-level number remains a useful diagnostic.


## 18. Error Analysis

TP/FP/FN/TN examples at the chosen threshold, with raw text fields alongside the computed
features, looking for recognizable failure patterns (near-duplicate names for different
businesses, abbreviations, missing addresses, multilingual text, same-country collisions,
overly aggressive blocking, etc.).


In [ ]:
final_preds = (valid_proba >= BEST_MACRO_THRESHOLD).astype(int)
valid_dataset = valid_dataset.copy()
valid_dataset["prediction"] = final_preds
valid_display = attach_record_fields(valid_dataset, conn)

display_cols = [
    "s1_business_name", "s1_business_address", "s1_country",
    "cand_business_name", "cand_business_address", "cand_country",
    "match_probability", "label", "prediction",
] + FEATURE_COLUMNS

true_positives = valid_display[(valid_display["label"] == 1) & (valid_display["prediction"] == 1)]
false_positives = valid_display[(valid_display["label"] == 0) & (valid_display["prediction"] == 1)]
false_negatives = valid_display[(valid_display["label"] == 1) & (valid_display["prediction"] == 0)]
true_negatives = valid_display[(valid_display["label"] == 0) & (valid_display["prediction"] == 0)]

print(f"TP={len(true_positives):,}  FP={len(false_positives):,}  "
      f"FN={len(false_negatives):,}  TN={len(true_negatives):,}")


In [ ]:
print("=== False Positives (highest-confidence mistakes) ===")
display(false_positives[display_cols].sort_values("match_probability", ascending=False).head(10))


In [ ]:
print("=== False Negatives (lowest-confidence mistakes) ===")
display(false_negatives[display_cols].sort_values("match_probability", ascending=True).head(10))


Go through the tables above and identify which recognizable pattern (if any) explains each
error — spelling variations, abbreviations, missing data, address formatting differences,
multilingual records, same-country/same-prefix collisions, or the blocking rules themselves
being too aggressive for a legitimate match. Summarize the dominant one or two patterns
actually observed once this has been run for real; don't just relist the category names.


## 19. France / Unseen-Country Distribution Discussion

The **test** set contains France, absent from **training**. We reason about this without
training on any test labels.

**Why `same_country` still works for an unseen country.** It's a *pairwise, relational*
feature — "do these two records claim the same country" — computed identically no matter
which country that is. A French/French pair produces the same feature value (`1`) as a
US/US pair, so the trained logistic-regression weight for `same_country` applies to it exactly
the same way; France doesn't need to have been seen for this feature to be meaningful.
`character_similarity` / `token_similarity` are likewise just string-overlap measures with no
fixed vocabulary, so they don't require having seen French text before either.

**What this does not establish.** The blocking prefix columns (`name_prefix`,
`address_prefix`) and the validation split above only reflect the training distribution's
countries. If French business names or addresses are conventionally formatted very
differently (accented first characters, legal-entity suffixes like `"SARL"`/`"SAS"`,
different word order), the *blocking rules themselves* could behave differently on French
data than the validation numbers suggest — this is a genuinely open question that only real
French validation/test data can answer, not something this notebook's synthetic-country-free
validation split can confirm either way.


## 20. Final Baseline Results

In [ ]:
pair_preds = (valid_proba >= BEST_MACRO_THRESHOLD).astype(int)
pair_precision = precision_score(y_valid, pair_preds, zero_division=0)
pair_recall = recall_score(y_valid, pair_preds, zero_division=0)
pair_f05 = fbeta_score(y_valid, pair_preds, beta=0.5, zero_division=0)

results_table = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Storage": "SQLite (database/amazon_ml.db)",
    "Blocking": "country+name_prefix OR country+address_prefix",
    "Batch size": BATCH_SIZE,
    "Negative Sampling": f"{NEGATIVE_TO_POSITIVE_RATIO}:1 (neg:pos), reservoir, seed={RANDOM_STATE}",
    "Class Weight": FINAL_CLASS_WEIGHT,
    "Threshold": BEST_MACRO_THRESHOLD,
    "Pair-level Precision": round(pair_precision, 4),
    "Pair-level Recall": round(pair_recall, 4),
    "Pair-level F0.5": round(pair_f05, 4),
    "Macro S1 F0.5": round(macro_f05, 4),
}])
display(results_table)

performance_report = pd.DataFrame([
    {"Metric": "Database file size (bytes)", "Value": DB_PATH.stat().st_size},
    {"Metric": "source1 rows", "Value": n_s1},
    {"Metric": "source2 rows", "Value": n_s2},
    {"Metric": "source3 rows", "Value": n_s3},
    {"Metric": "ground_truth_pairs rows (all positives)", "Value": n_positive_total},
    {"Metric": "Exact union candidate pairs (batched, deduped)", "Value": total_union_pairs},
    {"Metric": "Negative pairs sampled", "Value": len(reservoir)},
    {"Metric": "training_pairs total rows", "Value": n_train_positive + n_train_negative},
])
display(performance_report)


# 21. Conclusions — What Did This SQLite Baseline Teach Us?

*(Fill in the italicized placeholders with the actual numbers from a real run against
`student_resource/dataset` — nothing here is fabricated; treat this cell as a template, not a
final answer, until every placeholder is replaced with observed output.)*

**Database & blocking**
- Database size on disk: *see section 20's performance report*
- Candidate pairs implied by each rule (pre-union) and their exact de-duplicated union: *see
  sections 9 and 10*
- Share of ground-truth positives recoverable by live blocking alone (diagnostic, not a
  correctness requirement since positives are preserved regardless): *see section 9*

**Training data**
- Positive pairs: *`{n_positive_total}`* (ALL preserved, verified by assertion in section 11)
- Negative pairs sampled (ratio {NEGATIVE_TO_POSITIVE_RATIO}:1, seed {RANDOM_STATE}):
  *see section 10/11*

**Model & thresholds**
- Final configuration: Logistic Regression, `class_weight="{FINAL_CLASS_WEIGHT}"`, 11 features
- Best pair-level / macro-per-S1 thresholds: *see sections 16-17*

**Validation performance**
- Pair-level Precision / Recall / F0.5, and macro-per-S1 F0.5: *see section 20*

**Most informative features / dominant error patterns:** *fill in from sections 13 and 18.*

## What this version fixed vs. the previous (in-memory) baseline

1. **No more `MemoryError` from candidate generation.** Candidate pairs are never fully
   materialized in Python; they're generated and consumed batch-by-batch, with a bounded
   reservoir sample standing in for "all negatives."
2. **Blocking-rule sizing is now reportable *before* generating any pairs**, via cheap SQL
   `GROUP BY` aggregation over the small set of distinct blocking keys.
3. **Positive-pair preservation is now provably exact**, enforced by an `assert` comparing
   `ground_truth_pairs` and `training_pairs` counts, rather than relying on careful
   set-difference bookkeeping in Python.
4. **The database is reusable** — `RECREATE_DB = False` lets later sections (or later
   notebooks) skip straight to querying `training_pairs`/`source1/2/3` without re-importing
   or re-indexing.

## Remaining limitations (same spirit as the original baseline)

1. Multi-key blocking (country + first-PREFIX_LEN-chars of name/address) is still a hard
   recall ceiling for *live* candidate generation on genuinely new/unlabeled data — only
   training positives get a free pass via direct preservation.
2. Still only 11 hand-designed string-similarity features.
3. Negative sampling ratio is fixed, not calibrated against the true competition-scoring
   imbalance.
4. No hyperparameter tuning beyond the `class_weight` comparison.
5. French/unseen-country generalization is still argued conceptually (section 19), not
   measured directly — the validation split contains no France by construction.

## Possible future improvements

1. Smarter blocking (phonetic/soundex keys, longer or multiple prefix lengths, sorted-
   neighborhood blocking) evaluated the same way — via the section 9 SQL aggregate pattern —
   before committing to a change.
2. Hard-negative mining: bias the reservoir toward negatives that are textually close but
   wrong, rather than uniform random sampling.
3. Gradient boosting (XGBoost/LightGBM) over the same or an expanded feature set, still
   reading from `training_pairs`.
4. Multilingual embeddings / sentence-transformer similarity, particularly for the French
   generalization question.
5. Moving from pairwise S1-vs-candidate classification to graph-based clustering across all
   three sources at once.

This baseline is intentionally simple so each future change can be measured against a fixed,
reproducible, memory-safe reference point.


## Closing the Database Connection

Always close the SQLite connection when finished, so the `.db` file isn't left open/locked.


In [ ]:
conn.close()
print("SQLite connection closed.")
